# Project

## Introduction to the Project
The S&P 500 (Standard & Poor's 500) is a stock market index that tracks the performance of 500 of the largest publicly traded companies in the United States. It is widely regarded as one of the best representations of the U.S. stock market and economy. Over the long term, the S&P 500 has shown consistent growth, making it a key focus for long-term investors. However, it can also experience significant volatility in the short term.

In this project, we will make our first attempt to build a momentum-based strategy to trade the S&P 500 index. At the end of the project, you will have built a program that you can later expand and customise to suit your needs. We will use the python packages numpy, scipy and sqlite3, among others, in this project.

Tips: Review the code snippets that we went through during the course. Reorganize them and you are half-way done! Try experimenting with different configurations of the confidence interval, the lookback window and the forecast horizon. Be brave and experiment with different ways of deciding the position size. You may be surprised by your talent!

Re-organize your code from the exercises into a Python program that 
1. read prices from a database
2. calibrate a GBM model on each day when new prices are received.
3. forecast the price in e.g. 10 days and
   1. calculate the confidence interval of the forecast
   2. calculate the expected shortfall from the price forecast
4. code your trading signals using the price forecast and the expected shortfall.
5. store your postions into the *positions* table after each trade.
6. produce a 1-year backtest record from 2020-06-01 to 2021-05-31.

**Hint**

1. Collect all the code related to the GBM model into a class

In [ ]:
import csv
import sqlite3
from contextlib import closing
from datetime import datetime

import numpy as np
from scipy.stats import norm

In [ ]:
class GBM:
    def __init__(self, seed=None):
        self.mu = np.nan
        self.sigma = np.nan
        self.rng = np.random.default_rng(seed)

    def _assert_calibrated(self):
        if not np.isfinite(self.mu) or not np.isfinite(self.sigma):
            raise ValueError("Model must be calibrated before forecasting risk/returns")
        if self.sigma < 0:
            raise ValueError("sigma must be non-negative")

    def simulate(self, N, K, Dt, S0):
        if N <= 0 or K <= 0:
            raise ValueError("N and K must be positive")
        if Dt <= 0:
            raise ValueError("Dt must be positive")
        if S0 <= 0:
            raise ValueError("S0 must be positive")

        self._assert_calibrated()

        traj = np.full(shape=(N + 1, K), fill_value=np.nan)
        traj[0, :] = S0
        drift = (self.mu - self.sigma**2 / 2) * np.linspace(Dt, N * Dt, N)
        for i in range(K):
            W = np.cumsum(self.rng.normal(loc=0.0, scale=np.sqrt(Dt), size=N))
            traj[1:, i] = S0 * np.exp(drift + self.sigma * W)
        return traj

    def calibrate(self, trajectory, Dt):
        if Dt <= 0:
            raise ValueError("Dt must be positive")

        trajectory = np.asarray(trajectory, dtype=float)
        if trajectory.ndim != 1 or len(trajectory) < 2:
            raise ValueError("trajectory must be a 1D array with at least 2 points")
        if np.any(trajectory <= 0):
            raise ValueError("trajectory values must be strictly positive")

        increments = np.diff(np.log(trajectory))
        moments = [0.0, 0.0]
        n_iter = 10
        sample_size = max(2, len(increments) // 2)

        for _ in range(n_iter):
            X = self.rng.choice(increments, size=sample_size, replace=True)
            moments[0] += np.mean(X) / n_iter
            moments[1] += np.mean(X**2) / n_iter

        variance = max(0.0, moments[1] - moments[0] ** 2)
        std = np.sqrt(variance)
        self.sigma = std / np.sqrt(Dt)
        self.mu = moments[0] / Dt + self.sigma**2 / 2

    def forecast(self, latest, t, confidence):
        self._assert_calibrated()

        if latest <= 0:
            raise ValueError("latest must be positive")
        if t <= 0:
            raise ValueError("t must be positive")
        if not 0 < confidence < 1:
            raise ValueError("confidence must be between 0 and 1")

        predicted = latest * np.exp(self.mu * t)
        mu = (self.mu - self.sigma**2 / 2) * t
        sigma = self.sigma * np.sqrt(t)
        log_return_low, log_return_high = norm.ppf(
            [(1 - confidence) / 2, (1 + confidence) / 2], loc=mu, scale=sigma
        )
        price_low = latest * np.exp(log_return_low)
        price_high = latest * np.exp(log_return_high)
        return {
            "confidence": confidence,
            "expected": predicted,
            "interval": [price_low, price_high],
        }

    def expected_shortfall(self, T, confidence):
        self._assert_calibrated()

        if T <= 0:
            raise ValueError("T must be positive")
        if not 0 < confidence < 1:
            raise ValueError("confidence must be between 0 and 1")

        # Expected shortfall of log-return in the left tail.
        # Returned value is downside risk as a positive number.
        mu = (self.mu - self.sigma**2 / 2) * T
        sigma = self.sigma * np.sqrt(T)

        alpha = 1 - confidence
        z_alpha = norm.ppf(alpha)
        left_tail_mean = mu - sigma * norm.pdf(z_alpha) / alpha
        return -left_tail_mean

In [ ]:
# test your code here
model = GBM(seed=42)
model.mu = 0.3
model.sigma = 0.2
simulated = model.simulate(5000, 1, 1 / 250, 100)
simulated = simulated[:, 0]

model2 = GBM(seed=42)
model2.calibrate(simulated, 1 / 250)

print(f"Calibrated: mu = {model2.mu}, sigma = {model2.sigma}")

2. Write a function that prepares the database for trading, i.e.
   1. load the historical prices into the *prices* table
   2. create the *positions* table
   3. initialize the *positions* table with the your initial cash reserve. The initial *time_of_trade* can be any date before the earliest possible trading date.

    Call this function *prepare*.

In [ ]:
def prepare(reset_positions=True):
    with closing(sqlite3.connect("SP500.db")) as conn:
        cs = conn.cursor()

        cs.execute("""
        create table if not exists prices (
            theday text primary key,
            price real
        );
        """)

        with closing(open("SP500.csv")) as datafile:
            reader = csv.DictReader(
                datafile, fieldnames=["date", "price"], delimiter="\t"
            )
            for row in reader:
                cs.execute(
                    "insert or replace into prices values (?, ?)",
                    (row["date"], float(row["price"])),
                )
        print("Prices loaded into DB")

        cs.execute("""
        create table if not exists positions (
            time_of_trade text,
            instrument text,
            quantity real,
            cash real,
            primary key (time_of_trade, instrument)
        );
        """)
        print("Database ready")

        if reset_positions:
            cs.execute("delete from positions;")
            cs.execute(
                "insert or replace into positions values (?, ?, ?, ?);",
                ("1900-01-01", "SP500", 0.0, 100000.0),
            )
            print("Positions table seeded: 0 shares, $100,000 cash")

        conn.commit()


prepare()

In [ ]:
# check whether you have loaded the prices correctly
# conn and cs are kept open as shared globals used by the rest of the program
conn = sqlite3.connect("SP500.db")
cs = conn.cursor()
latest_prices = cs.execute("select * from prices order by theday desc limit 10")
for item in latest_prices:
    print(item)

3. Write a function that determines the trade size, i.e. how many units of the instrument you would like to own when the date is *which_day* and the price forecast of the instrument is *forecast* and the expected shortfall from the same forecast is *ES*.

In [ ]:
def position_size(which_day, forecast, ES):
    cs.execute(f"""
    select quantity, cash from positions
    where instrument = 'SP500'
    and time_of_trade < '{which_day}'
    order by time_of_trade desc
    limit 1;
    """)
    qty, cash = cs.fetchall()[0]
    cs.execute(f"""
    select price from prices
    where theday <= '{which_day}'
    order by theday desc
    limit 1;
    """)
    price = cs.fetchall()[0][0]
    capital = cash + qty * price
    exposure = min(capital * 0.05 / ES, capital)
    expected = forecast["expected"]
    if expected > price:
        return round(exposure / price)
    elif expected < price:
        return -round(exposure / price)
    else:
        return 0

In [ ]:
# quick smoke test for position_size
# price on 2021-05-14 is 4179

# expected (4150) < price (4179) → model predicts decline → SELL
test_forecast = {"interval": [4000.0, 4300.0], "expected": 4150.0, "confidence": 0.95}
print(position_size("2021-05-14", test_forecast, 0.02))

# expected (4300) > price (4179) → model predicts rise → BUY
test_forecast_buy = {
    "interval": [4200.0, 4400.0],
    "expected": 4300.0,
    "confidence": 0.95,
}
print(position_size("2021-05-14", test_forecast_buy, 0.02))

# expected (4000) < price (4179) → model predicts decline → SELL
test_forecast_sell = {
    "interval": [3900.0, 4100.0],
    "expected": 4000.0,
    "confidence": 0.95,
}
print(position_size("2021-05-14", test_forecast_sell, 0.02))

4. Write a function that, for a given date, calibrates a GBM model to the data prior to that date and that forecasts the price in 10 days. Call this function *analyse*.

In [ ]:
def analyse(which_day):
    cs.execute(f"""
    select price from prices where theday <= '{which_day}'
    order by theday desc limit 120;
    """)
    P = np.flipud(np.asarray(cs.fetchall())).flatten()
    model = GBM()
    Dt = 1.0 / 252
    model.calibrate(P, Dt)
    confidence = 0.9
    n = 10
    T = n * Dt
    forecast = model.forecast(P[-1], T, confidence)
    ES = model.expected_shortfall(T, confidence)
    return position_size(which_day, forecast, ES)

In [ ]:
# Test the analyse function
test_dates = ["2021-05-09", "2021-05-14"]
positions = [np.nan, np.nan]
for i in range(2):
    positions[i] = analyse(test_dates[i])
    print(f"{positions[i]} shares advised on {test_dates[i]}.")

5. The main loop of the program: Loop over the dates in the backtest period and use the *analyse* function to decide what to do on each day. Call this function *main*.

In [ ]:
def main(begin_on):
    cs.execute(f"select theday from prices where theday >= '{begin_on}';")
    days = [d[0] for d in cs.fetchall()]
    asset = {"old": np.nan, "new": np.nan}
    cash = {"old": np.nan, "new": np.nan}
    cs.execute("delete from positions where time_of_trade > '2020-01-01';")
    for d in days:
        asset["new"] = analyse(d)
        cs.execute(f"""
        select quantity, cash from positions
        where time_of_trade < '{d}'
        order by time_of_trade desc
        limit 1;
        """)
        asset["old"], cash["old"] = cs.fetchall()[0]
        cs.execute(f"""
        select price from prices
        where theday <= '{d}'
        order by theday desc
        limit 1;
        """)
        latest = cs.fetchall()[0][0]
        trade_size = round(asset["new"]) - round(asset["old"])
        if trade_size != 0:
            cash["new"] = cash["old"] - trade_size * latest
            cs.execute(f"""
            insert into positions values
            ('{d}', 'SP500', {round(asset['new'])}, {cash['new']});
            """)
        conn.commit();

6. Connect to the database and create a *cursor* object associated with the connection. Share the connection and the cursor object across the program so that you don't have to connect to and disconnect from the database in every function of the program.

In [ ]:
prepare()
main("2020-06-01")

In [ ]:
# plot your track record
conn = sqlite3.connect("SP500.db")
cs = conn.cursor()

day1 = "2020-06-01"
day1_dt = datetime.strptime(day1, "%Y-%m-%d")

cs.execute(f"""
    select theday, quantity * price + cash as wealth
    from positions as PO
    join prices as PR
    on PO.time_of_trade = (
        select time_of_trade from positions
        where time_of_trade <= PR.theday
        order by time_of_trade desc limit 1
    )
    where theday >= '{day1}';
""")

records = cs.fetchall()


def calculate_T(record, day1_dt):
    theday, wealth = record
    theday_dt = datetime.strptime(theday, "%Y-%m-%d")
    T = (theday_dt - day1_dt).days
    return (T, wealth)


records = [calculate_T(record, day1_dt) for record in records]
W = np.asarray(records)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = plt.plot(W[:, 0], W[:, 1])
plt.title("S&P 500 Momentum Strategy — 1-Year Backtest (2020-06-01 to 2021-05-31)")
plt.grid()
plt.xlabel("Number of days of trading")
plt.ylabel("Total Wealth");

In [ ]:
# Buy-and-hold benchmark: invest $100k on 2020-06-01, sell on 2021-05-31
conn_bh = sqlite3.connect("SP500.db")
cs_bh = conn_bh.cursor()
cs_bh.execute("select price from prices where theday = '2020-06-01';")
price_start = cs_bh.fetchone()[0]
cs_bh.execute("select price from prices where theday = '2021-05-31';")
price_end = cs_bh.fetchone()[0]
conn_bh.close()

initial_capital = 100_000.0
shares_bought = initial_capital / price_start
final_value = shares_bought * price_end
profit = final_value - initial_capital
ret = (price_end / price_start - 1) * 100

print(f"Start price (2020-06-01): ${price_start:,.2f}")
print(f"End price   (2021-05-31): ${price_end:,.2f}")
print(f"Shares bought: {shares_bought:.4f}")
print(f"Final value:   ${final_value:,.2f}")
print(f"Profit:        ${profit:,.2f}  ({ret:.1f}%)")

In [ ]:
from scipy.stats import ttest_1samp

conn_t = sqlite3.connect("SP500.db")
cs_t = conn_t.cursor()
cs_t.execute("""
    select price from prices
    where theday >= '2020-06-01' and theday <= '2021-05-31'
    order by theday asc;
""")
prices_bt = np.array([r[0] for r in cs_t.fetchall()])
conn_t.close()

log_returns = np.diff(np.log(prices_bt))
t_stat, p_value = ttest_1samp(log_returns, popmean=0)

print("Student's t-test on daily log-returns (H0: mean return = 0)")
print(
    f"  Mean daily log-return : {log_returns.mean():.6f}  ({log_returns.mean()*252:.2%} annualized)"
)
print(f"  t-statistic           : {t_stat:.4f}")
print(f"  p-value               : {p_value:.4f}")
if p_value < 0.05:
    print("  => Momentum is statistically significant at the 5% level.")
elif p_value < 0.10:
    print("  => Momentum is significant at the 10% level only.")
else:
    print("  => Momentum is NOT statistically significant (cannot reject H0).")

In [ ]:
# Daily wealth is in W: column 0 = days elapsed, column 1 = total wealth
daily_pnl = np.diff(W[:, 1])
daily_days = W[1:, 0].astype(int)

# Map days-elapsed back to calendar dates
conn_d = sqlite3.connect("SP500.db")
cs_d = conn_d.cursor()
cs_d.execute(
    "select theday from prices where theday >= '2020-06-01' order by theday asc;"
)
trading_dates = [r[0] for r in cs_d.fetchall()]
conn_d.close()

# Match W dates to trading dates by days-elapsed
day1_dt = datetime.strptime("2020-06-01", "%Y-%m-%d")
date_map = {(datetime.strptime(d, "%Y-%m-%d") - day1_dt).days: d for d in trading_dates}
pnl_dates = [date_map.get(int(d), "?") for d in daily_days]

best_idx = np.argmax(daily_pnl)
worst_idx = np.argmin(daily_pnl)

print("Best trading day:")
print(f"  Date   : {pnl_dates[best_idx]}")
print(f"  P&L    : ${daily_pnl[best_idx]:,.2f}")
print()
print("Worst trading day:")
print(f"  Date   : {pnl_dates[worst_idx]}")
print(f"  P&L    : ${daily_pnl[worst_idx]:,.2f}")

In [ ]:
# Daily strategy returns from wealth series
strategy_returns = np.diff(W[:, 1]) / W[:-1, 1]

mean_daily = strategy_returns.mean()
std_daily = strategy_returns.std(ddof=1)

# Risk-free rate: US savings account ~0.5% annually in 2020-2021
annual_risk_free = 0.005
daily_risk_free = annual_risk_free / 252

sharpe = (mean_daily - daily_risk_free) / std_daily * np.sqrt(252)

print("Sharpe Ratio Analysis")
print(f"  Mean daily return      : {mean_daily:.6f}  ({mean_daily*252:.2%} annualized)")
print(
    f"  Std  daily return      : {std_daily:.6f}  ({std_daily*np.sqrt(252):.2%} annualized vol)"
)
print(f"  Risk-free rate         : {annual_risk_free:.2%} per year (savings account)")
print(f"  Sharpe ratio           : {sharpe:.4f}")
print()
if sharpe > 1.0:
    print("  => Good risk-adjusted return (Sharpe > 1).")
elif sharpe > 0:
    print("  => Positive but modest risk-adjusted return (0 < Sharpe < 1).")
else:
    print(
        "  => Strategy did not outperform the risk-free rate on a risk-adjusted basis."
    )